# Isopropylamine

In [3]:
using Clapeyron, Metaheuristics, Printf

In [4]:
like_parameter = """Clapeyron Database File
PCSAFT Like Parameters [csvtype = like,grouptype = PCSAFT]
species,Mw,segment,sigma,epsilon,n_H,n_e
isopropylamine,59.11,2.68622,3.440089484,228.4542964,1,1
"""

assoc_parameter = """Clapeyron Database File
PCSAFT Assoc Parameters [csvtype = assoc,grouptype = PCSAFT]
species1,site1,species2,site2,epsilon_assoc,bondvol
isopropylamine,H,isopropylamine,e,800.0,0.05
"""

model = PCSAFT(["isopropylamine"], userlocations = [like_parameter, assoc_parameter])

print(model.params.epsilon_assoc.values)

Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[800.0]

In [5]:
# Run this ONCE to fix your CSV files
function fix_line_endings(filename)
    content = read(filename, String)
    fixed = replace(content, "\r\n" => "\n")
    write(filename, fixed)
    println("Fixed: $filename")
end

fix_line_endings("satp_isopropylamine.csv")
fix_line_endings("rhol_isopropylamine.csv")

Fixed: satp_isopropylamine.csv
Fixed: rhol_isopropylamine.csv


In [6]:
### Saturation pressure — output in Pa (SI)
function my_saturation_p(model::EoSModel, T::Float64)
    sat = saturation_pressure(model, T)
    return sat[1]   # Pa
end

# Liquid density — output in kg/m³
function my_liquid_density(model::EoSModel, T::Float64)
    sat  = saturation_pressure(model, T)
    Mw   = model.params.Mw[1]      # g/mol
    rhol = 1.0 / sat[2]            # mol/m³
    return rhol * Mw / 1000.0      # mol/m³ × g/mol ÷ 1000 = kg/m³
end

println(my_liquid_density(model, 293.15))
println(my_saturation_p(model, 273.15))

690.4363331927423
21377.730312375086


In [7]:
toestimate = [
    Dict(
        :param   => :epsilon_assoc,
        :indices => (1,1),
        :lower   => 0.0,
        :upper   => 1000.0,
        :guess   => 750.0
    ),
    Dict(
        :param   => :bondvol,
        :indices => (1,1),
        :lower   => 0.001,
        :upper   => 1.0,
        :guess   => 0.05
    )
]

2-element Vector{Dict{Symbol, Any}}:
 Dict(:upper => 1000.0, :param => :epsilon_assoc, :indices => (1, 1), :guess => 750.0, :lower => 0.0)
 Dict(:upper => 1.0, :param => :bondvol, :indices => (1, 1), :guess => 0.05, :lower => 0.001)

In [8]:
estimator, objective, x0, upper, lower = Estimation(
    model,
    toestimate,
    [
        "rhol_isopropylamine.csv",
        "satp_isopropylamine.csv",
    ]
)
 
println("Initial objective value: ", objective(x0))

Initial objective value: 0.010801288096118375


In [9]:
method = ECA(; options = Options(iterations = 10000000, seed = 999))
 
params_opt, model_opt = optimize(objective, estimator, method)

([927.4465182336857, 0.017366826523334558], PCSAFT{BasicIdeal, Float64}("isopropylamine"))

In [10]:
using CSV, DataFrames, Printf

function calculate_AAD(model, csv_file, property_func)
    df = CSV.read(csv_file, DataFrame, comment="#", skipto=4)
    
    input_col  = names(df)[1]          # first column = input (T)
    output_col = names(df)[2]          # second column = out_xxx (experimental)
    
    inputs   = df[!, input_col]
    exp_vals = df[!, output_col]
    
    println("\n=== AAD: $csv_file ===")
    @printf("%-10s  %-12s  %-12s  %-8s\n", input_col, "exp", "calc", "ARD%")
    
    errors = Float64[]
    for (i, x) in enumerate(inputs)
        calc = property_func(model, x)
        err  = abs(calc - exp_vals[i]) / abs(exp_vals[i]) * 100
        push!(errors, err)
        @printf("%-10.4f  %-12.6f  %-12.6f  %-8.4f\n", x, exp_vals[i], calc, err)
    end
    
    aard = sum(errors) / length(errors)
    @printf("AARD = %.4f%%\n", aard)
    return aard
end

calculate_AAD (generic function with 1 method)

In [11]:
aard_p   = calculate_AAD(model_opt, "satp_isopropylamine.csv",   my_saturation_p)


=== AAD: satp_isopropylamine.csv ===

┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593



Clapeyron Estimator  exp           calc          ARD%    
213.0900    440.497000    436.851703    0.8275  
217.7200    666.612000    663.449841    0.4744  
222.5300    1001.118000   1000.724979   0.0393  
227.8600    1533.074000   1538.360932   0.3449  
232.7300    2217.951000   2229.771352   0.5329  
237.7400    3182.538000   3201.766925   0.6042  
242.6100    4442.701000   4469.464509   0.6024  
281.6200    38547.500000  38599.874927  0.1359  
286.2000    47358.770000  47380.300224  0.0455  
290.8200    57803.250000  57786.370478  0.0292  
295.4800    70108.900000  70046.508528  0.0890  
300.1800    84525.050000  84410.068680  0.1360  
304.9100    101325.000000  101110.190099  0.2120  
309.6800    120798.070000  120466.914893  0.2741  
314.4900    143268.200000  142794.742427  0.3305  
319.3400    169052.800000  168431.606736  0.3675  
324.2200    198530.300000  197675.811183  0.4304  
329.1500    232087.600000  231033.272395  0.4543  
334.1100    270111.100000  268779.069068  0.493

0.3380542467841055

In [12]:
aard_p   = calculate_AAD(model_opt, "rhol_isopropylamine.csv",   my_liquid_density)


=== AAD: rhol_isopropylamine.csv ===
Clapeyron Estimator  exp           calc          ARD%    
213.1500    771.300000    766.766944    0.5877  
233.1500    751.300000    745.862092    0.7238  
253.1500    731.200000    725.079445    0.8371  
273.1500    710.300000    704.170633    0.8629  
293.1500    689.100000    682.863975    0.9050  
313.1500    666.500000    660.849348    0.8478  
333.1500    643.200000    637.757746    0.8461  
353.1500    618.400000    613.128090    0.8525  
373.1500    592.400000    586.348282    1.0216  
AARD = 0.8316%


┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593


0.8316055403370134